# 02 — Mətn Ön Emalı (Preprocessing)

Apar — İstifadəçi Rəylərinin Təsnifatı

**EDA-dan əsas tapıntılar:**
- Label paylanması: `technical_support` 59.89%, `other` 21.39%, `customer_support` 18.72% → **imbalance var**
- `customer_support` rəyləri ortalama ~57 simvol, digərləri ~31-32 simvol → **uzunluq ayırd edici feature**
- `tag` sütununda yalnız **29.6%** dolub (12504 NaN) → **has_tag binary feature** əlavə edilir

In [ ]:
import pandas as pd
import re
import sys
sys.path.append('../src')
from preprocess import clean_text, build_features

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

## Mətn Temizleme

In [ ]:
train['clean_feedback'] = train['feedback'].apply(clean_text)
test['clean_feedback']  = test['feedback'].apply(clean_text)
train[['feedback','clean_feedback']].head(10)

## Əlavə Feature-lər (EDA əsasında)

EDA göstərdi ki:
1. `customer_support` rəyləri ~57 simvol, digərləri ~31-32 simvol — **`feedback_len`** ayırd edici ola bilər
2. `tag` sütunu yalnız 29.6% dolub — **`has_tag`** (0/1) modelin bu siqnalı istifadə etməsinə imkan verir

In [ ]:
# feedback uzunluğu
train['feedback_len'] = train['feedback'].fillna('').apply(len)
test['feedback_len']  = test['feedback'].fillna('').apply(len)

# tag mövcudluğu (binary)
train['has_tag'] = train['tag'].notna().astype(int)
test['has_tag']  = test['tag'].notna().astype(int)

print('feedback_len — label üzrə ortalama:')
print(train.groupby('label')['feedback_len'].mean().round(1))
print()
print('has_tag — label üzrə paylanma:')
print(train.groupby('label')['has_tag'].mean().round(3))

## Birləşdirilmiş Feature (feedback + tag)

In [ ]:
train['text'] = train.apply(build_features, axis=1)
test['text']  = test.apply(build_features, axis=1)
train[['text','label']].head(10)

## Class Imbalance Qeydi

EDA-dan: `technical_support` 59.89%, `other` 21.39%, `customer_support` 18.72%.

Model qurarkən tövsiyə olunur:
- `class_weight='balanced'` (sklearn modellərində)
- və ya `stratify=y` ilə train/val split

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = train['label'].unique()
weights = compute_class_weight('balanced', classes=classes, y=train['label'])
class_weight_dict = dict(zip(classes, weights))
print('Class weights:', class_weight_dict)

## Faylları Saxla

In [ ]:
train.to_csv('../data/train_processed.csv', index=False)
test.to_csv('../data/test_processed.csv', index=False)
print('Hazırdır!')